# 07. Recovery curve and return/displacement analysis

This notebook Computes:

- **weekly cumulative return curves** for directly impacted (`burnt`) and precautionary / non-directly-impacted (`not-burnt`) evacuees, corresponding to **Figure 4a**
- the association between **social connectedness** and **social similarity** of evacuation destinations and later return/displacement outcomes, corresponding to **Figure 4b**
- the week-specific return regressions reported in the **Supplementary Tables S21–S28**

## Required input files
This notebook expects the same processed files used in the original working notebook:

- `weekly_raw_noid`
- `evac_n_disp.csv.gz`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

In [ ]:
# ------------------------------------------------------------------
# Input files
# ------------------------------------------------------------------
WEEKLY_HOME_PATH = "weekly_raw_noid"
EVAC_DISP_PATH = "evac_n_disp.csv.gz"

# ------------------------------------------------------------------
# Output files
# ------------------------------------------------------------------
FIG4A_OUT = "figure4a_cumulative_return_curves.png"
FIG4B_OUT = "figure4b_return_regression_coefficients.png"
TABLE_OUT = "return_regression_results_week1_week20.csv"

weekly_home = pd.read_csv(WEEKLY_HOME_PATH)
evac_disp = pd.read_csv(EVAC_DISP_PATH)

weekly_home.head()

In [ ]:
# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------
WEEK_COLS = [f"week{i}" for i in range(1, 23)]

def standardize_cbg_string(series):
    """Convert identifiers to string and strip trailing '.0' artifacts."""
    return (
        series.astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .replace({"nan": np.nan, "None": np.nan})
    )

def find_return_week(row, home_col="pre_crisis_home_cbg", week_cols=WEEK_COLS):
    """Return the first week in which the individual is observed back in the pre-crisis home CBG."""
    for week in week_cols:
        if pd.notna(row[week]) and row[week] == row[home_col]:
            return week
    return np.nan

def week_to_numeric(week_label):
    if pd.isna(week_label):
        return np.nan
    return int(str(week_label).replace("week", ""))

def cumulative_return_curve(df, total_n, week_range=range(1, 23)):
    """Cumulative percentage returned, keeping non-returners in the denominator."""
    returned = df.dropna(subset=["return_week_numeric"])
    weekly_counts = (
        returned["return_week_numeric"]
        .value_counts()
        .reindex(week_range, fill_value=0)
        .sort_index()
    )
    weekly_pct = (weekly_counts / total_n) * 100
    return weekly_pct.cumsum()

def bootstrap_cumulative_return(df, total_n, week_range=range(1, 23), n_bootstrap=1000, random_state=42):
    """Bootstrap 95% CI for the cumulative return curve."""
    rng = np.random.default_rng(random_state)
    boot = []
    for _ in range(n_bootstrap):
        sample_idx = rng.integers(0, len(df), size=len(df))
        boot_df = df.iloc[sample_idx]
        curve = cumulative_return_curve(boot_df, total_n=total_n, week_range=week_range)
        boot.append(curve.values)
    boot = pd.DataFrame(boot, columns=list(week_range))
    lower = boot.quantile(0.025, axis=0)
    upper = boot.quantile(0.975, axis=0)
    return lower, upper

def fit_standardized_logit(df, y_col, x_col):
    """Fit a one-predictor logit with standardized x and return model outputs."""
    tmp = df[[y_col, x_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if tmp.empty or tmp[y_col].nunique() < 2:
        return None

    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(tmp[[x_col]])
    X = pd.DataFrame(x_scaled, columns=[x_col], index=tmp.index)
    X = sm.add_constant(X)

    model = sm.Logit(tmp[y_col].astype(int), X)
    result = model.fit(disp=0)
    return result

In [ ]:
# ------------------------------------------------------------------
# Clean identifiers and compute first observed return week
# ------------------------------------------------------------------
df = weekly_home.copy()

df["pre_crisis_home_cbg"] = standardize_cbg_string(df["pre_crisis_home_cbg"])
df["crisis_home_cbg"] = standardize_cbg_string(df["crisis_home_cbg"])

for col in WEEK_COLS:
    df[col] = standardize_cbg_string(df[col])

df["return_week"] = df.apply(find_return_week, axis=1)
df["return_week_numeric"] = df["return_week"].apply(week_to_numeric)

df[["pre_crisis_home_cbg", "crisis_home_cbg", "impact", "return_week", "return_week_numeric"]].head()

In [ ]:
# ------------------------------------------------------------------
# Merge return outcomes with homophily / connectedness measures
# ------------------------------------------------------------------
evac_disp["pre_crisis_home_cbg_x"] = standardize_cbg_string(evac_disp["pre_crisis_home_cbg_x"])
evac_disp["crisis_home_cbg"] = standardize_cbg_string(evac_disp["crisis_home_cbg"])
evac_disp["last_known_post_crisis_home_cbg"] = standardize_cbg_string(evac_disp["last_known_post_crisis_home_cbg"])

analysis_df = pd.merge(
    df,
    evac_disp,
    left_on=["pre_crisis_home_cbg", "crisis_home_cbg"],
    right_on=["pre_crisis_home_cbg_x", "crisis_home_cbg"],
    how="inner"
)

analysis_df["returned_anytime"] = analysis_df["return_week_numeric"].notna().astype(int)
analysis_df["log_scaled_sci"] = np.log1p(analysis_df["scaled_sci"])
analysis_df["log_disp_scaled_sci"] = np.log1p(analysis_df["disp_scaled_sci"])

analysis_df.shape

In [ ]:
# ------------------------------------------------------------------
# Figure 4a: cumulative return curves with bootstrap 95% CI
# ------------------------------------------------------------------
burnt_df = analysis_df[analysis_df["impact"] == "burnt"].copy()
not_burnt_df = analysis_df[analysis_df["impact"] == "not-burnt"].copy()

week_range = range(1, 23)

burnt_curve = cumulative_return_curve(burnt_df, total_n=len(burnt_df), week_range=week_range)
not_burnt_curve = cumulative_return_curve(not_burnt_df, total_n=len(not_burnt_df), week_range=week_range)

burnt_low, burnt_high = bootstrap_cumulative_return(burnt_df, total_n=len(burnt_df), week_range=week_range)
not_burnt_low, not_burnt_high = bootstrap_cumulative_return(not_burnt_df, total_n=len(not_burnt_df), week_range=week_range)

plt.figure(figsize=(10, 5))

plt.errorbar(
    burnt_curve.index, burnt_curve.values,
    yerr=[burnt_curve.values - burnt_low.values, burnt_high.values - burnt_curve.values],
    fmt="o", capsize=4, elinewidth=1.2, label="Directly impacted (burnt)"
)
plt.plot(burnt_curve.index, burnt_curve.values)

plt.errorbar(
    not_burnt_curve.index, not_burnt_curve.values,
    yerr=[not_burnt_curve.values - not_burnt_low.values, not_burnt_high.values - not_burnt_curve.values],
    fmt="x", capsize=4, elinewidth=1.2, label="Precautionary / not-burnt"
)
plt.plot(not_burnt_curve.index, not_burnt_curve.values)

plt.xticks(list(week_range))
plt.xlabel("Weeks after disaster")
plt.ylabel("Cumulative return percentage (%)")
plt.grid(True, axis="y", linestyle="--", linewidth=0.5)
plt.legend(frameon=False)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(FIG4A_OUT, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Create week-specific return indicators used in Supplementary Tables S21-S28
# ------------------------------------------------------------------
for week in ["week1", "week20"]:
    analysis_df[f"returned_{week}"] = (
        analysis_df["pre_crisis_home_cbg"] == analysis_df[week]
    ).astype(int)

analysis_df[["impact", "returned_week1", "returned_week20", "log_scaled_sci", "log_disp_scaled_sci", "disp_homophily"]].head()

In [ ]:
# ------------------------------------------------------------------
# Logistic regressions for week 1 and week 20 return outcomes
# Predictors follow the structure used in the original notebook:
# - week1  : log_scaled_sci, disp_homophily
# - week20 : log_disp_scaled_sci, disp_homophily
# ------------------------------------------------------------------
predictors = {
    "week1": ["log_scaled_sci", "disp_homophily"],
    "week20": ["log_disp_scaled_sci", "disp_homophily"],
}

group_map = {
    "affected": "burnt",
    "non_affected": "not-burnt",
}

results = {}
rows = []

for week, x_cols in predictors.items():
    y_col = f"returned_{week}"

    for group_name, impact_value in group_map.items():
        sub = analysis_df[analysis_df["impact"] == impact_value].copy()

        for x_col in x_cols:
            res = fit_standardized_logit(sub, y_col=y_col, x_col=x_col)
            results[(week, group_name, x_col)] = res

            if res is not None:
                rows.append({
                    "week": week,
                    "group": group_name,
                    "predictor": x_col,
                    "coef": res.params[x_col],
                    "p_value": res.pvalues[x_col],
                    "ci_low": res.conf_int().loc[x_col, 0],
                    "ci_high": res.conf_int().loc[x_col, 1],
                    "n_obs": int(res.nobs),
                    "pseudo_r2": float(res.prsquared),
                })

results_df = pd.DataFrame(rows)
results_df.to_csv(TABLE_OUT, index=False)
results_df

In [ ]:
# ------------------------------------------------------------------
# Figure 4b-style coefficient plot for the return/displacement regressions
# ------------------------------------------------------------------
plot_df = results_df.copy()

label_map = {
    "log_scaled_sci": "Connectedness (week 1)",
    "log_disp_scaled_sci": "Connectedness (week 20)",
    "disp_homophily": "Similarity"
}
plot_df["predictor_label"] = plot_df["predictor"].map(label_map)

# order: affected first, then non_affected
plot_df["group_order"] = plot_df["group"].map({"affected": 0, "non_affected": 1})
plot_df = plot_df.sort_values(["group_order", "week", "predictor_label"]).reset_index(drop=True)

x = np.arange(len(plot_df))

plt.figure(figsize=(8, 5))
for i, row in plot_df.iterrows():
    plt.errorbar(
        x=i,
        y=row["coef"],
        yerr=[[row["coef"] - row["ci_low"]], [row["ci_high"] - row["coef"]]],
        fmt="o",
        capsize=4
    )

plt.axhline(0, linestyle="--", linewidth=1)
plt.xticks(
    x,
    [f"{g}\n{w}\n{p}" for g, w, p in zip(plot_df["group"], plot_df["week"], plot_df["predictor_label"])],
    rotation=0
)
plt.ylabel("Standardized logit coefficient")
plt.xlabel("")
plt.grid(True, axis="y", linestyle="--", linewidth=0.5)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(FIG4B_OUT, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Print compact model summaries
# ------------------------------------------------------------------
for key, res in results.items():
    print("=" * 90)
    print(key)
    if res is None:
        print("Model could not be estimated.")
    else:
        print(res.summary())